# Fig. S1B, S1C: agreement between statistical methods

Each calibration assay is shown two ways: a padj-vs-padj scatter coloured by gene density (the converging mass on y=x) and a rank-rank hexbin (agreement of the orderings), both annotated with Spearman rho and (where a y=x calibration applies) concordance. S2c is a three-way Venn. The small net excess of permutation hits indicates that the chi-sq counts are conservative.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, gaussian_kde

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig1/supp_2"
B = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/DirMult_optimized")
THR = 0.05
DENS_CMAP = "Blues"  # gene density (light -> dark) on the diagonal = the methods converge


def bh(p):
    p = np.asarray(p, float)
    ok = np.isfinite(p)
    out = np.full(p.shape, np.nan)
    pp = p[ok]
    n = len(pp)
    o = np.argsort(pp)
    q = np.minimum.accumulate((pp[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    res = np.empty(n)
    res[o] = np.clip(q, 0, 1)
    out[ok] = res
    return out


def density_scatter(ax, xv, yv):
    """Scatter coloured by 2-D point density (KDE; histogram fallback)."""
    xv = np.asarray(xv, float)
    yv = np.asarray(yv, float)
    try:
        dens = gaussian_kde(np.vstack([xv, yv]))(np.vstack([xv, yv]))
    except Exception:
        H, xe, ye = np.histogram2d(xv, yv, bins=50)
        ix = np.clip(np.digitize(xv, xe) - 1, 0, H.shape[0] - 1)
        iy = np.clip(np.digitize(yv, ye) - 1, 0, H.shape[1] - 1)
        dens = H[ix, iy]
    o = np.argsort(dens)  # dense points drawn on top
    return ax.scatter(
        xv[o], yv[o], c=dens[o], cmap=DENS_CMAP, s=5, alpha=0.85, linewidths=0, rasterized=True
    )


def rank_hexbin(ax, pa, pb):
    """Hexbin of the two methods' gene rankings (density)."""
    rx = pd.Series(np.asarray(pa, float)).rank(method="average").values
    ry = pd.Series(np.asarray(pb, float)).rank(method="average").values
    hb = ax.hexbin(rx, ry, gridsize=42, cmap=DENS_CMAP, bins="log", mincnt=1, linewidths=0)
    ax.plot([rx.min(), rx.max()], [ry.min(), ry.max()], color="0.5", lw=0.8, zorder=5)
    return hb


def annot(ax, rho, extra=None):
    txt = f"Spearman $\\rho$ = {rho:.2f}" + (
        f"\nconcordance {extra:.1f}%" if extra is not None else ""
    )
    ax.text(
        0.04,
        0.97,
        txt,
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT,
        va="top",
        ha="left",
        fontweight="bold",
    )


def finish(fig, ax, xlab, ylab, title, cb_obj=None, cb_label="point density"):
    if cb_obj is not None:
        cb = fig.colorbar(cb_obj, ax=ax, fraction=0.045, pad=0.02)
        cb.set_label(cb_label, fontsize=S.FONT_TICK)
        cb.ax.tick_params(labelsize=S.FONT_TICK - 1)
    ax.set_xlabel(xlab, fontsize=S.FONT_LABEL)
    ax.set_ylabel(ylab, fontsize=S.FONT_LABEL)
    ax.set_title(title, fontsize=S.FONT_ANNOT, pad=3)
    ax.tick_params(labelsize=S.FONT_TICK)
    ax.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(left=0.16, right=0.93, top=0.90, bottom=0.13)

In [ ]:
# S2a - permutation vs chi-squared (padj concordance, density-coloured)
perm = pd.read_csv(
    B
    / "permutation_batch_number"
    / "dirmult_regression_permutation_optimized2_joint_df5_pt-dpt_pseudotime_cov-batch_number.csv"
)
d = perm.copy()  # all multi-site genes (broad calibration)
d["padj_chi"] = bh(d.p_value_chisq.values)
d["padj_perm"] = bh(d.p_value_perm.values)
d = d.dropna(subset=["padj_chi", "padj_perm", "p_value_chisq", "p_value_perm"])
rho = spearmanr(d.p_value_chisq, d.p_value_perm).correlation
conc = 100 * (((d.padj_chi < THR) == (d.padj_perm < THR)).mean())
gained = int(((d.padj_perm < THR) & (d.padj_chi >= THR)).sum())
lost = int(((d.padj_perm >= THR) & (d.padj_chi < THR)).sum())
print(f"S2a perm: rho={rho:.3f} conc={conc:.1f}% +{gained}/-{lost}")
xclip = min(-np.log10(float(d.padj_perm.min())), 15)
xv = (-np.log10(d.padj_chi.clip(lower=1e-300))).clip(upper=xclip).values
yv = (-np.log10(d.padj_perm.clip(lower=1e-300))).clip(upper=xclip).values
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 780 / 300))
sc = density_scatter(ax, xv, yv)
ax.plot([0, xclip], [0, xclip], color="0.5", lw=0.8, zorder=6)
t = -np.log10(THR)
ax.axhline(t, color="k", lw=0.4, ls=":")
ax.axvline(t, color="k", lw=0.4, ls=":")
ax.set_xlim(0, xclip * 1.03)
ax.set_ylim(0, xclip * 1.03)
annot(ax, rho, conc)
ax.text(
    0.98,
    0.04,
    f"net +{gained - lost} toward permutation\n($\\chi^2$ conservative; perm padj discrete)",
    transform=ax.transAxes,
    fontsize=S.FONT_ANNOT - 1.5,
    va="bottom",
    ha="right",
    color="0.5",
    style="italic",
)
finish(
    fig,
    ax,
    r"$-\log_{10}$ padj  ($\chi^2$ LRT)",
    r"$-\log_{10}$ padj  (permutation)",
    "permutation null calibrates the $\\chi^2$ LRT",
    sc,
)
S.save_and_show(fig, "fig_s2a_permutation_vs_chisq", figure=FIG)

In [ ]:
# S2b - pseudobulk DRIMSeq vs cell-level (padj concordance, density-coloured; matches S2a/S2d)
pb = pd.read_csv(B / "pseudobulk_pseudotime" / "drimseq_gene_all_pseudotime.csv")
comp = pd.read_csv(B / "composite_ranking_optimized2_df5.csv")
keyR = "gene_id" if pb["gene_id"].astype(str).str.startswith("ENSMUS").any() else "gene_name"
m = pb.merge(
    comp[[keyR, "padj"]].rename(columns={keyR: "gene_id"}), on="gene_id", how="inner"
).dropna(subset=["adj_pvalue", "padj"])
rho = spearmanr(m.padj, m.adj_pvalue).correlation
pb_only = int(((m.adj_pvalue < THR) & (m.padj >= THR)).sum())
print(f"S2b pseudobulk: rho={rho:.3f} pb_only={pb_only} n={len(m)}")
xv = (-np.log10(m.padj.clip(lower=1e-300))).clip(upper=50).values
yv = (-np.log10(m.adj_pvalue.clip(lower=1e-300))).clip(upper=20).values
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 780 / 300))
sc = density_scatter(ax, xv, yv)
ax.plot([0, 20], [0, 20], color="0.5", lw=0.8, zorder=6)
ax.axhline(-np.log10(THR), color="k", lw=0.4, ls=":")
ax.axvline(-np.log10(THR), color="k", lw=0.4, ls=":")
ax.set_xlim(0, 51)
ax.set_ylim(0, 21)
annot(ax, rho)
ax.text(
    0.98,
    0.04,
    f"every pseudobulk hit is also cell-level\n({pb_only} pseudobulk-only of {len(m):,} testable)",
    transform=ax.transAxes,
    fontsize=S.FONT_ANNOT - 1.5,
    va="bottom",
    ha="right",
    color="0.5",
    style="italic",
)
finish(
    fig,
    ax,
    r"$-\log_{10}$ padj  (cell-level DM LRT)",
    r"$-\log_{10}$ padj  (pseudobulk DRIMSeq)",
    "pseudobulk confirms a strict subset",
    sc,
)
S.save_and_show(fig, "fig_s2b_pseudobulk_vs_celllevel", figure=FIG)

In [ ]:
# S2c - three-way Venn (chi-sq / permutation / pseudobulk)
from matplotlib_venn import venn3, venn3_circles

tw = pd.read_csv(B / "three_way_defensibility.csv").iloc[0]
subsets = (
    int(tw.n_chisq_only),
    int(tw.n_perm_only),
    int(tw.n_chisq_and_perm_not_pb),
    int(tw.n_pb_only),
    int(tw.n_chisq_and_pb_not_perm),
    int(tw.n_perm_and_pb_not_chisq),
    int(tw.n_all_three),
)
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 760 / 300))
v = venn3(
    subsets=subsets,
    set_labels=("chi-sq LRT", "permutation", "pseudobulk"),
    ax=ax,
    set_colors=("#2166AC", "#1b7837", "#B2182B"),
    alpha=0.45,
)
venn3_circles(subsets=subsets, ax=ax, linewidth=0.6, color="0.4")
for tt in v.set_labels or []:
    if tt:
        tt.set_fontsize(S.FONT_LABEL)
for tt in v.subset_labels or []:
    if tt:
        tt.set_fontsize(S.FONT_TICK)
ax.set_title(
    f"pseudobulk hits are a strict subset of chi-sq AND permutation\n"
    f"({int(tw.n_pb_only)} pseudobulk-only, of {int(tw.n_three_way_tested):,} testable genes)",
    fontsize=S.FONT_ANNOT,
    pad=2,
)
fig.subplots_adjust(left=0.04, right=0.96, top=0.86, bottom=0.04)
S.save_and_show(fig, "fig_s2c_three_way_venn", figure=FIG)

In [ ]:
# S2d - parametric bootstrap vs chi-squared (padj concordance, density-coloured; secondary)
comp = pd.read_csv(B / "composite_ranking_optimized2_df5.csv")
bs = comp[comp.p_value_boot.notna()].copy()
bs["padj_chi"] = bs["padj"].values  # Main chi-sq FDR (not re-BH on the subset)
bs["padj_boot"] = bh(bs.p_value_boot.values)
bs = bs.dropna(subset=["padj_chi", "padj_boot", "p_value_chisq", "p_value_boot"])
rho = spearmanr(bs.p_value_chisq, bs.p_value_boot).correlation
conc = 100 * (((bs.padj_chi < THR) == (bs.padj_boot < THR)).mean())
print(f"S2d boot: rho={rho:.3f} conc={conc:.1f}% n={len(bs)}")
xclip = min(-np.log10(float(bs.padj_boot.min())), 15)
xv = (-np.log10(bs.padj_chi.clip(lower=1e-300))).clip(upper=xclip).values
yv = (-np.log10(bs.padj_boot.clip(lower=1e-300))).clip(upper=xclip).values
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 780 / 300))
sc = density_scatter(ax, xv, yv)
ax.plot([0, xclip], [0, xclip], color="0.5", lw=0.8, zorder=6)
ax.axhline(-np.log10(THR), color="k", lw=0.4, ls=":")
ax.axvline(-np.log10(THR), color="k", lw=0.4, ls=":")
ax.set_xlim(0, xclip * 1.03)
ax.set_ylim(0, xclip * 1.03)
annot(ax, rho, conc)
ax.text(
    0.98,
    0.04,
    "secondary diagnostic\n(superseded by permutation)",
    transform=ax.transAxes,
    fontsize=S.FONT_ANNOT - 1.5,
    va="bottom",
    ha="right",
    color="0.5",
    style="italic",
)
finish(
    fig,
    ax,
    r"$-\log_{10}$ padj  ($\chi^2$ LRT)",
    r"$-\log_{10}$ padj  (parametric bootstrap)",
    f"bootstrap vs $\\chi^2$ (borderline subset, n={len(bs):,})",
    sc,
)
S.save_and_show(fig, "fig_s2d_bootstrap_vs_chisq", figure=FIG)

In [ ]:
# S2e - rank-rank concordance: permutation vs chi-squared (hexbin density)
d2 = perm.dropna(subset=["p_value_chisq", "p_value_perm"]).copy()
rho = spearmanr(d2.p_value_chisq, d2.p_value_perm).correlation
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 780 / 300))
hb = rank_hexbin(ax, d2.p_value_chisq, d2.p_value_perm)
annot(ax, rho)
finish(
    fig,
    ax,
    r"gene rank  ($\chi^2$ LRT)",
    "gene rank  (permutation)",
    "rank-rank: permutation vs $\\chi^2$",
    hb,
    cb_label="genes",
)
S.save_and_show(fig, "fig_s2e_rank_permutation", figure=FIG)

In [ ]:
# S2f - rank-rank concordance: pseudobulk vs cell-level (hexbin density)
rho = spearmanr(m.padj, m.adj_pvalue).correlation
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 780 / 300))
hb = rank_hexbin(ax, m.padj, m.adj_pvalue)
annot(ax, rho)
finish(
    fig,
    ax,
    "gene rank  (cell-level DM LRT)",
    "gene rank  (pseudobulk)",
    "rank-rank: pseudobulk vs cell-level",
    hb,
    cb_label="genes",
)
S.save_and_show(fig, "fig_s2f_rank_pseudobulk", figure=FIG)

In [ ]:
# S2g - rank-rank concordance: bootstrap vs chi-squared (hexbin density)
rho = spearmanr(bs.p_value_chisq, bs.p_value_boot).correlation
fig, ax = plt.subplots(figsize=(S.HALF_W_PX / 300, 780 / 300))
hb = rank_hexbin(ax, bs.p_value_chisq, bs.p_value_boot)
annot(ax, rho)
finish(
    fig,
    ax,
    r"gene rank  ($\chi^2$ LRT)",
    "gene rank  (bootstrap)",
    "rank-rank: bootstrap vs $\\chi^2$",
    hb,
    cb_label="genes",
)
S.save_and_show(fig, "fig_s2g_rank_bootstrap", figure=FIG)